# Strain & mosaicity analysis — 111 strain-mosa (Al, 17 keV)

Inputs are the outputs of `process_strainmosa.py`:

- `mean.npy` — shape `(Y, X, 3)`, the intensity-weighted centre of the strongest 3D peak per pixel. Channels are **phi, chi, obpitch**.
- `motors.npy` — shape `(3, m, n, o)`, the motor coordinate grid (phi, chi, obpitch).

**Physics.** This is dark-field X-ray microscopy (Poulsen *et al.*, *J. Appl. Cryst.* **50**, 1441–1456, 2017). The two sample tilts (phi, chi) map local **mosaicity** (lattice orientation), and a scan of the objective angle **obpitch** scans the scattering angle 2θ, which maps the **axial strain** ε = Δd/d.

**Geometric strain gradient.** The measured 2θ has a geometric offset that is *linear in the vertical sample coordinate* z (Poulsen eqs 27–28: `2θ_shift = −γ·z_s`), experimentally γ ≈ 2.93 m⁻¹ (their Fig. 6). In a reconstructed map the vertical sample coordinate maps to the image **row** axis, so the raw 2θ/strain map carries a linear ramp along rows that is *not* real strain and must be removed before interpreting strain. Note the 2θ shift depends on z only (the horizontal coordinate y feeds the roll/η component, not 2θ), so the correction is a **vertical** de-ramp.

Strain conversion (Bragg, fixed λ): ε = Δd/d = −cot(θ_B)·Δθ, with Δθ = Δ(2θ)/2.

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import darling

# ----------------------------------------------------------------------- #
# Paths
# ----------------------------------------------------------------------- #
DATA_DIR = Path(
    "~/Documents/Data/4dcells/111_june/"
    "111_cells_2_6-7pct_strainmosa_2x_redo2/strainmosa_3d"
).expanduser()

# ----------------------------------------------------------------------- #
# Channel mapping  (VERIFY against the printed motor ranges below:
# phi/chi are small sample tilts; obpitch ~ 17.9-18.0 deg)
# ----------------------------------------------------------------------- #
CH_PHI, CH_CHI, CH_OBP = 0, 1, 2
CH_LABELS = {CH_PHI: "phi", CH_CHI: "chi", CH_OBP: "obpitch (2theta)"}

# ----------------------------------------------------------------------- #
# Crystal / beam: Al (fcc) 111 at 17 keV
# ----------------------------------------------------------------------- #
A_AL = 4.0495        # Angstrom, room-T lattice parameter
E_KEV = 17.0
HC = 12.39842        # keV * Angstrom

# obpitch step == 2theta step (degrees). Verified: 2theta_B ~ 17.946 deg
# matches the obpitch scan start (17.9441 deg), i.e. obpitch is 2theta in deg.
OBPITCH_TO_2THETA = 1.0

# Axis of the geometric strain ramp. Geometry predicts purely vertical (rows).
RAMP_AXIS = 0        # 0 = rows (vertical); switch to 1 only if the ramp is horizontal


In [ ]:
mean = np.load(DATA_DIR / "mean.npy")        # (Y, X, 3)
motors = np.load(DATA_DIR / "motors.npy")    # (3, m, n, o)
try:
    info = json.loads((DATA_DIR / "processing_info.json").read_text())
except FileNotFoundError:
    info = {}
mean = mean[150:-150, 100:-200, :]
print("mean   :", mean.shape)
print("motors :", motors.shape)
for c, lbl in CH_LABELS.items():
    print(f"  ch {c} = {lbl:18s} motor grid range [{motors[c].min():.4f}, {motors[c].max():.4f}]")
print("obpitch steps:", np.round(motors[CH_OBP, 0, 0, :], 5))


In [ ]:
# --- Bragg-angle sanity check: 2theta_B should sit at the start of the obpitch scan
d111 = A_AL / np.sqrt(3.0)
lam = HC / E_KEV
theta_B = np.degrees(np.arcsin(lam / (2.0 * d111)))
cot_B = 1.0 / np.tan(np.radians(theta_B))

print(f"d111     = {d111:.4f} A")
print(f"lambda   = {lam:.4f} A")
print(f"theta_B  = {theta_B:.4f} deg")
print(f"2*theta_B= {2*theta_B:.4f} deg")
print(f"cot(theta_B) = {cot_B:.4f}")
print(f"obpitch range = [{motors[CH_OBP].min():.4f}, {motors[CH_OBP].max():.4f}] deg")


In [ ]:
# --- Valid-pixel mask: peaks leaves unlabelled pixels at 0, which is far
# --- outside the obpitch range, so a range check cleanly drops the background.
obp = mean[..., CH_OBP]
obp_lo, obp_hi = motors[CH_OBP].min(), motors[CH_OBP].max()
valid = (obp >= obp_lo - 1e-6) & (obp <= obp_hi + 1e-6)
print(f"valid pixels: {valid.mean()*100:.1f} %")


In [ ]:
v = np.nanpercentile(np.abs(strain_m), 98)# --- Raw centroid maps: phi, chi, obpitch
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (c, lbl) in zip(axes, CH_LABELS.items()):
    m = np.where(valid, mean[..., c], np.nan)
    im = ax.imshow(m, cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_title(lbl)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## Mosaicity map (phi–chi)

The two sample tilts combined into an orientation (RGB) map via `darling.transforms.rgb`. `norm="full"` scales colours to the full scanned phi/chi range, so colours are comparable across datasets that share the same mosa grid.

In [ ]:
mosa = mean[..., [CH_PHI, CH_CHI]].astype(float)
mosa[~valid] = np.nan

rgb_map, colorkey, colorgrid = darling.transforms.rgb(
    mosa, norm="full", coordinates=motors[[CH_PHI, CH_CHI]]
)

fig, ax = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [3, 1]})
ax[0].imshow(rgb_map, aspect=2.8)
ax[0].set_title("Mosaicity (phi-chi) RGB")
ax[1].imshow(
    colorkey, origin="lower", aspect="auto",
    extent=[motors[CH_PHI].min(), motors[CH_PHI].max(),
            motors[CH_CHI].min(), motors[CH_CHI].max()],
)
ax[1].set_xlabel("phi"); ax[1].set_ylabel("chi"); ax[1].set_title("colour key")
plt.tight_layout()
plt.show()


## Axial strain: remove the geometric vertical ramp, then convert

Per Poulsen eqs (27)–(28), the apparent 2θ varies linearly with vertical sample position (image rows); this is geometry, not strain. We fit a plane to the obpitch map over valid pixels, report the row and column slopes (geometry expects the row slope to dominate), and subtract the **row** ramp only.

**Limitation (be aware):** an empirical vertical de-ramp also removes any *real* uniform vertical strain gradient — the two are degenerate without the calibrated γ. To preserve a real vertical gradient you must subtract the *theoretical* ramp γ·z_s instead (needs the magnification M, the sample-to-objective distance / γ, and the effective pixel size for this Al setup). Hook for that is in the cell below.

In [ ]:
rr, cc = np.where(valid)
Amat = np.vstack([rr, cc, np.ones_like(rr)]).T
(srow, scol, b0), *_ = np.linalg.lstsq(Amat, obp[valid], rcond=None)

print(f"row (vertical) slope = {srow:.3e} deg/px  -> {srow*mean.shape[0]:.4f} deg across FOV")
print(f"col (horizontal) slope = {scol:.3e} deg/px  -> {scol*mean.shape[1]:.4f} deg across FOV")
print("Geometry (eq 27-28): 2theta shift is purely vertical -> expect |row| >> |col|.")

# --- empirical geometric de-ramp (default) ---
if RAMP_AXIS == 0:
    ramp = srow * np.arange(mean.shape[0])[:, None]
else:
    ramp = scol * np.arange(mean.shape[1])[None, :]

# --- theoretical alternative (uncomment + fill in to preserve real vertical gradients):
# GAMMA = 2.93           # m^-1, geometric factor (Poulsen Fig 6 value is setup-specific!)
# M_MAG = 15.1           # total magnification
# PIX_M = 0.65e-6 * 2    # effective pixel size at sample (detector pixel * bin / M); set for your setup
# z_s = -(np.arange(mean.shape[0]) - mean.shape[0]/2)[:, None] * PIX_M
# ramp = -np.degrees(GAMMA * z_s)   # 2theta_shift = -gamma * z_s

obp_corr = obp - ramp
obp_corr_m = np.where(valid, obp_corr, np.nan)


In [ ]:
# --- visualise the ramp removal as a row profile (median over columns)
rows = np.arange(mean.shape[0])
prof_raw = np.nanmedian(np.where(valid, obp, np.nan), axis=1)
prof_cor = np.nanmedian(obp_corr_m, axis=1)

plt.figure(figsize=(6, 5))
plt.plot(prof_raw, rows, label="raw")
plt.plot(prof_cor, rows, label="de-ramped")
plt.gca().invert_yaxis()
plt.xlabel("obpitch / 2theta (deg)"); plt.ylabel("row")
plt.title("Vertical geometric ramp removal"); plt.legend()
plt.show()


In [ ]:
# --- convert corrected 2theta to axial strain  eps = -cot(theta_B) * d(2theta)/2
two_theta = obp_corr * OBPITCH_TO_2THETA                          # deg
two_theta_ref = np.nanmedian(np.where(valid, two_theta, np.nan))  # relative zero (FOV median)
dtheta = np.radians(two_theta - two_theta_ref) / 2.0              # rad
strain = -cot_B * dtheta
strain_m = np.where(valid, strain, np.nan)

print(f"reference 2theta = {two_theta_ref:.4f} deg (strain zero)")
print(f"strain rms = {np.nanstd(strain_m):.2e}")
print(f"strain 2-98 pct = [{np.nanpercentile(strain_m,2):.2e}, {np.nanpercentile(strain_m,98):.2e}]")


In [ ]:
# --- strain map (diverging colour scale, symmetric)
v = np.nanpercentile(np.abs(strain_m), 98)
plt.figure(figsize=(7, 6))
im = plt.imshow(strain_m, cmap="RdBu_r", vmin=-v, vmax=v, aspect=2.8)
plt.colorbar(im, label=r"axial strain $\varepsilon = \Delta d / d$")
plt.title("Axial strain")
plt.show()


In [ ]:
from pathlib import Path

# --- save strain map as NumPy array
out_path = Path.home() / "Desktop" / "single_xtal_6p7pct_strain.npy"
np.save(out_path, strain_m)

print(f"Saved strain map to: {out_path}")

# --- plot with correct geometric aspect ratio
aspect_ratio = 1.0 / np.tan(np.radians(17.9))
print(f"imshow aspect ratio = {aspect_ratio:.3f}")

v = np.nanpercentile(np.abs(strain_m), 98)

plt.figure(figsize=(7, 6))
im = plt.imshow(
    strain_m,
    cmap="RdBu_r",
    vmin=-v,
    vmax=v,
    aspect=aspect_ratio,
)
plt.colorbar(im, label=r"axial strain $\varepsilon = \Delta d / d$")
plt.title("Axial strain")
plt.tight_layout()
plt.show()

## Assumptions & limitations

- **Channel order** is taken as `phi, chi, obpitch`. Verify against the printed motor ranges (obpitch must be the ~17.9–18.0° channel). If phi/chi are swapped, swap `CH_PHI`/`CH_CHI`.
- **obpitch = 2θ in degrees, 1:1.** Supported by 2θ_B = 17.946° matching the obpitch start (17.9441°). If the instrument applies a different obpitch→2θ scaling, set `OBPITCH_TO_2THETA`.
- **Geometric correction** removes the vertical (row) ramp, which is what Poulsen eqs (27)–(28) predict for the 2θ shift (linear in vertical sample coordinate z; horizontal y affects roll/η, not 2θ). The empirical de-ramp cannot distinguish this from a real uniform vertical strain gradient — use the theoretical-γ hook if you need to keep real vertical gradients.
- **θ_B** uses the room-temperature Al lattice parameter (4.0495 Å); a deformed/strained sample shifts this slightly, which only rescales the absolute strain, not the maps.
- Strain here is **relative** (zero at the FOV median 2θ). For absolute strain set the reference to an unstrained region or to 2θ_B.

Reference: H. F. Poulsen *et al.*, *J. Appl. Cryst.* **50**, 1441–1456 (2017) — eqs (3), (27)–(28), Fig. 6.